In [1]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import random

def load_imdb_subset(size=3000):   # smaller for speed
    dataset = load_dataset("imdb")

    texts = dataset["train"]["text"]
    labels = dataset["train"]["label"]

    combined = list(zip(texts, labels))
    random.shuffle(combined)

    texts, labels = zip(*combined)

    texts = list(texts[:size])
    labels = list(labels[:size])

    return train_test_split(texts, labels, test_size=0.2, random_state=42)

X_train, X_test, y_train, y_test = load_imdb_subset()

# Load data
X_train, X_test, y_train, y_test = load_imdb_subset()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Naive Bayes
vec_nb = CountVectorizer(max_features=10000)
X_nb = vec_nb.fit_transform(X_train)
nb = MultinomialNB().fit(X_nb, y_train)

pred_nb = nb.predict(vec_nb.transform(X_test))

# Logistic Regression
vec_lr = TfidfVectorizer(max_features=10000)
X_lr = vec_lr.fit_transform(X_train)
lr = LogisticRegression(max_iter=1000).fit(X_lr, y_train)

pred_lr = lr.predict(vec_lr.transform(X_test))

def evaluate(name, y_true, y_pred):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("F1:", f1_score(y_true, y_pred))

evaluate("Naive Bayes", y_test, pred_nb)
evaluate("Logistic Regression", y_test, pred_lr)


Naive Bayes
Accuracy: 0.8383333333333334
F1: 0.8289241622574955

Logistic Regression
Accuracy: 0.8433333333333334
F1: 0.8469055374592834


In [3]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_enc = tokenizer(X_train, truncation=True, padding=True)
test_enc = tokenizer(X_test, truncation=True, padding=True)

class Dataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_ds = Dataset(train_enc, y_train)
test_ds = Dataset(test_enc, y_test)

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds
)

trainer.train()

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,0.547885
100,0.339962
150,0.333662


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=150, training_loss=0.40716946919759117, metrics={'train_runtime': 127.2392, 'train_samples_per_second': 18.862, 'train_steps_per_second': 1.179, 'total_flos': 317921756774400.0, 'train_loss': 0.40716946919759117, 'epoch': 1.0})

In [4]:
preds = trainer.predict(test_ds).predictions.argmax(-1)

evaluate("DistilBERT", y_test, preds)


DistilBERT
Accuracy: 0.8883333333333333
F1: 0.8885191347753744


In [26]:
test_cases = [
    # ---- CATEGORY 1: ALL SHOULD PASS ----
    ("This is a good movie", 1),
    ("This is a bad movie", 0),
    ("Amazing acting and great story", 1),
    ("Terrible plot and boring film", 0),

    # ---- CATEGORY 2: NB vs LR behavior ----
    ("It was a long, long, long, long, long movie, but it was great.", 1),

    # ---- CATEGORY 3: neutral words takes as neg in nb and lr ----
    ("The cinematography was adequate, the sound was passable, and the acting was fine", 1),

    # ---- CATEGORY 4: STRUCTURE (BERT advantage) ----
    ("Bad start but excellent ending saves the film", 1),
    ("I expected this to be good, but it turned out to be great", 1),

    # ---- CATEGORY 5: HARD CASES (ALL FAIL) ----
    ("The prequel was terrible but this movie is actually good", 1),
    ("The actor's success streak broke with this film", 0),
    ("The story made up for the poor acting and weak plot", 1),
]

for text, label in test_cases:
    nb_pred = nb.predict(vec_nb.transform([text]))[0]
    lr_pred = lr.predict(vec_lr.transform([text]))[0]

    inputs = tokenizer([text], return_tensors="pt", truncation=True, padding=True).to(model.device)
    bert_pred = model(**inputs).logits.argmax(-1).item()

    print(f"\n{text}")
    print(f"True: {label} | NB: {nb_pred} | LR: {lr_pred} | BERT: {bert_pred}")


This is a good movie
True: 1 | NB: 0 | LR: 1 | BERT: 1

This is a bad movie
True: 0 | NB: 0 | LR: 0 | BERT: 0

Amazing acting and great story
True: 1 | NB: 1 | LR: 1 | BERT: 1

Terrible plot and boring film
True: 0 | NB: 0 | LR: 0 | BERT: 0

It was a long, long, long, long, long movie, but it was great.
True: 1 | NB: 0 | LR: 1 | BERT: 1

The cinematography was adequate, the sound was passable, and the acting was fine
True: 1 | NB: 0 | LR: 0 | BERT: 1

Bad start but excellent ending saves the film
True: 1 | NB: 1 | LR: 0 | BERT: 1

I expected this to be good, but it turned out to be great
True: 1 | NB: 0 | LR: 1 | BERT: 1

The prequel was terrible but this movie is actually good
True: 1 | NB: 0 | LR: 0 | BERT: 0

The actor's success streak broke with this film
True: 0 | NB: 1 | LR: 1 | BERT: 1

The actor's failure streak broke with this film
True: 1 | NB: 0 | LR: 0 | BERT: 0

The story made up for the poor acting and weak plot
True: 1 | NB: 0 | LR: 0 | BERT: 0

What a masterpiece... no